In [6]:
import numpy as np
import math
from collections import deque
from scipy.optimize import nnls


def normalize(root):
    return tuple(round(float(c), 6) for c in root)


def compute_roots_up_to_depth(M, max_depth, max_iter=8000):
    r = len(M)
    
    B = np.zeros((r, r))
    for i in range(r):
        B[i, i] = 1.0
        for j in range(i+1, r):
            m = M[i][j]
            if m == float('inf'):
                B[i, j] = -1.0
                B[j, i] = -1.0
            else:
                B[i, j] = -math.cos(math.pi / m)
                B[j, i] = B[i, j]
    
    roots = {}
    for i in range(r):
        coeffs = normalize(tuple(1 if j == i else 0 for j in range(r)))
        roots[coeffs] = 0
    
    queue = deque(roots.keys())
    iteration = 0
    
    while queue and iteration < max_iter:
        iteration += 1
        coeffs = queue.popleft()
        depth = roots[coeffs]
        
        if depth >= max_depth:
            continue
        
        beta = np.array(coeffs)
        
        for i in range(r):
            alpha = np.zeros(r)
            alpha[i] = 1
            
            B_ab = np.dot(alpha, np.dot(B, beta))
            new_beta = beta - 2 * B_ab * alpha
            new_coeffs = normalize(tuple(np.round(new_beta, decimals=8)))
            
            if not all(c >= -1e-8 for c in new_coeffs):
                continue
            if all(abs(c) < 1e-8 for c in new_coeffs):
                continue
            
            if B_ab <= -1 + 1e-9:
                new_depth = depth + 1
            else:
                new_depth = depth
            
            if new_depth <= max_depth:
                if new_coeffs not in roots:
                    roots[new_coeffs] = new_depth
                    queue.append(new_coeffs)
                elif new_depth < roots[new_coeffs]:
                    roots[new_coeffs] = new_depth
                    queue.append(new_coeffs)
    
    unique_roots = {}
    for coeffs, depth in roots.items():
        key = normalize(coeffs)
        if key not in unique_roots or depth < unique_roots[key]:
            unique_roots[key] = depth
    
    return unique_roots, B


def compute_inversion_set(word, B):
    r = len(B)
    inv_set = set()
    
    for idx, s in enumerate(word):
        alpha = np.zeros(r)
        alpha[s] = 1
        beta = alpha.copy()
        
        for j in range(idx-1, -1, -1):
            t = word[j]
            a = np.zeros(r)
            a[t] = 1
            B_ab = np.dot(a, np.dot(B, beta))
            beta = beta - 2 * B_ab * a
        
        inv_set.add(normalize(tuple(np.round(beta, decimals=8))))
    
    return inv_set


def get_root_depth(root, roots_depth):
    for r, d in roots_depth.items():
        if len(r) == len(root) and all(abs(r[i] - root[i]) < 1e-4 for i in range(len(root))):
            return d
    return None


def is_root_in_cone(root, small_roots):
    if not small_roots:
        return False
    
    root = np.array(root)
    small_roots = [np.array(sr) for sr in small_roots]
    
    for sr in small_roots:
        if np.linalg.norm(root - sr) < 1e-4:
            return True
    
    A = np.array([sr for sr in small_roots]).T
    
    try:
        c, residual = nnls(A, root)
        if residual < 1e-3:
            return True
        reconstructed = A @ c
        if np.linalg.norm(reconstructed - root) < 1e-3:
            return True
    except:
        pass
    
    return False


def is_low_of_layer(inv_set, roots_depth, layer):
    if len(inv_set) == 0:
        return True
    
    small_in_inv = []
    for root in inv_set:
        depth = get_root_depth(root, roots_depth)
        if depth is None:
            return False
        if depth <= layer:
            small_in_inv.append(root)
    
    if not small_in_inv:
        return False
    
    for root in inv_set:
        if not is_root_in_cone(root, small_in_inv):
            return False
    
    return True


def compute_low_elements(M, target_layer, max_iter=8000, max_depth=20): #####################
    r = len(M)
    
    roots_depth, B = compute_roots_up_to_depth(M, target_layer + 15, max_iter)##################
    
    low_elements = {}
    inv_to_word = {}
    layer_words = {i: [] for i in range(target_layer + 1)}
    
    # e
    low_elements[()] = set()
    inv_to_word[tuple(sorted(set()))] = ()
    if 0 <= target_layer:
        layer_words[0].append(())
    
    # Simple roots
    for i in range(r):
        word = (i,)
        inv_set = compute_inversion_set(word, B)
        if inv_set is not None and is_low_of_layer(inv_set, roots_depth, 0):
            low_elements[word] = inv_set
            key = tuple(sorted(inv_set))
            if key not in inv_to_word:
                inv_to_word[key] = word
                if 0 <= target_layer:
                    layer_words[0].append(word)
    
    def dfs(word, last, current_layer):
        if len(word) >= max_depth:
            return
        
        for s in range(r):
            if len(word) > 0 and s == word[0]:
                continue
            
            new_word = (s,) + word
            inv_set = compute_inversion_set(new_word, B)
            if inv_set is None:
                continue
            
            layers = [is_low_of_layer(inv_set, roots_depth, i) for i in range(target_layer + 1)]
            
            #if len(new_word) >= 5:
               # layer_str = ", ".join([f"layer{i}={layers[i]}" for i in range(min(3, len(layers)))])
               # print(f"word={word_to_string(new_word)}, len={len(new_word)}, {layer_str}")
            
            for layer_idx in range(target_layer + 1):
                if layers[layer_idx]:
                    layer_words[layer_idx].append(new_word)
                    key = tuple(sorted(inv_set))
                    if key not in inv_to_word:
                        inv_to_word[key] = new_word
                        low_elements[new_word] = inv_set
            
            if is_low_of_layer(inv_set, roots_depth, current_layer):
                dfs(new_word, s, current_layer)
            elif is_low_of_layer(inv_set, roots_depth, current_layer + 1):
                if current_layer + 1 <= target_layer:
                    dfs(new_word, s, current_layer + 1)
    
    for i in range(r):
        if (i,) in low_elements:
            dfs((i,), i, 0)
    
    return low_elements, layer_words


def word_to_string(word):
    if not word:
        return 'e'
    return ''.join(str(i+1) for i in word)


def print_low_elements(M, target_layer):
    r = len(M)
    print(f"\nComputing {target_layer}-low elements...")
    low_elements, layer_words = compute_low_elements(M, target_layer)
    
    all_words = layer_words.get(target_layer, [])
    
    # Add e and the simple roots
    all_words = [()] + [(i,) for i in range(r)] + all_words
    
    inv_to_word_final = {}
    for word in all_words:
        if word in low_elements:
            inv_set = low_elements[word]
            key = tuple(sorted(inv_set))
            if key not in inv_to_word_final or len(word) < len(inv_to_word_final[key]):
                inv_to_word_final[key] = word
    
    final_words = list(inv_to_word_final.values())
    final_words.sort(key=lambda x: (len(x), x))
    
    print(f"\nFound {len(final_words)} {target_layer}-low elements:")
    print("-" * 60)
    
    for word in final_words:
        print(f"  {word_to_string(word)}")
    
    print("-" * 60)


if __name__ == "__main__":
    print("=" * 70)
    print("Low Elements Calculator")
    print("=" * 70)
    print("\nEnter the Coxeter matrix (separate entries in each row with spaces; enter 'done' to finish):")
    
    M = []
    while True:
        line = input().strip()
        if line.lower() == 'done':
            break
        if line:
            row = []
            for val in line.split():
                if val.lower() == 'inf':
                    row.append(float('inf'))
                else:
                    row.append(float(val))
            M.append(row)
    
    if not M:
        print("No matrix entered. Exiting.")
        exit()
    
    target_layer = int(input("\nEnter the target layer n: "))
    print_low_elements(M, target_layer)

Low Elements Calculator

Enter the Coxeter matrix (separate entries in each row with spaces; enter 'done' to finish):


 1 4 4
 4 1 4
 4 4 1
 done

Enter the target layer n:  1



Computing 1-low elements...

Found 40 1-low elements:
------------------------------------------------------------
  e
  1
  2
  3
  12
  13
  21
  23
  31
  32
  121
  123
  131
  132
  212
  213
  231
  232
  312
  313
  321
  323
  1232
  1323
  2121
  2131
  2313
  3121
  3131
  3212
  3232
  13232
  23131
  32121
  123131
  132121
  213232
  232121
  313232
  323131
------------------------------------------------------------
